In [1]:
# load fundq

import pandas as pd

fundq = pd.read_csv("../data/fundamental/fundamental_quartly.csv")

In [2]:
import pandas as pd
import numpy as np

def build_fundamental_features(fundq_raw):
    df = fundq_raw.copy()

    # ------------------------
    # Basic cleaning
    # ------------------------
    df["datadate"] = pd.to_datetime(df["datadate"])
    df = df.sort_values(["gvkey", "datadate"])

    # Only standard consolidated industry statements
    df = df[
        (df["datafmt"] == "STD") &
        (df["indfmt"] == "INDL") &
        (df["consol"] == "C")
    ].copy()

    # ------------------------
    # 1. Lag variables
    # ------------------------
    lag1_cols = ["revtq", "atq", "uceqq"]
    for col in lag1_cols:
        df[f"{col}_lag1"] = df.groupby("gvkey")[col].shift(1)
        df[f"{col}_lag4"] = df.groupby("gvkey")[col].shift(4)

    # ------------------------
    # 2. Trailing 12-month (TTM) flows
    # ------------------------
    flow_cols = ["revtq", "niq", "oiadpq", "cogsq", "xsgaq", "oancfy", "capxy"]
    for col in flow_cols:
        df[f"{col}_ttm"] = (
            df.groupby("gvkey")[col]
              .transform(lambda x: x.rolling(4, min_periods=1).sum())
        )

    # Short aliases
    df["revt_ttm"]   = df["revtq_ttm"]
    df["ni_ttm"]     = df["niq_ttm"]
    df["oiadp_ttm"]  = df["oiadpq_ttm"]
    df["cogs_ttm"]   = df["cogsq_ttm"]
    df["xsga_ttm"]   = df["xsgaq_ttm"]
    df["oancf_ttm"]  = df["oancfy_ttm"]
    df["capx_ttm"]   = df["capxy_ttm"]

    # ------------------------
    # 3. Growth features
    # ------------------------
    df["sales_growth_qoq"] = df["revtq"] / df["revtq_lag1"] - 1

    df["revt_ttm_lag4"] = df.groupby("gvkey")["revt_ttm"].shift(4)
    df["sales_growth_ttm"] = df["revt_ttm"] / df["revt_ttm_lag4"] - 1

    df["asset_growth"] = df["atq"] / df["atq_lag4"] - 1
    df["equity_growth"] = df["uceqq"] / df["uceqq_lag4"] - 1

    # ------------------------
    # 4. Profitability
    # ------------------------
    df["roa_ttm"] = df["ni_ttm"] / df["atq"]
    df["roe_ttm"] = df["ni_ttm"] / df["uceqq"]

    df["gross_margin_ttm"] = (df["revt_ttm"] - df["cogs_ttm"]) / df["revt_ttm"]
    df["oper_margin_ttm"]  = df["oiadp_ttm"] / df["revt_ttm"]
    df["net_margin_ttm"]   = df["ni_ttm"] / df["revt_ttm"]

    # ------------------------
    # 5. Valuation
    # ------------------------
    df["log_mktcap"] = np.log(df["mkvaltq"].clip(lower=1))

    df["bm"] = df["uceqq"] / df["mkvaltq"]
    df["earnings_yield"] = df["ni_ttm"] / df["mkvaltq"]
    df["cf_yield"]       = df["oancf_ttm"] / df["mkvaltq"]
    df["sales_yield"]    = df["revt_ttm"] / df["mkvaltq"]

    df["div_yield"] = (df["dvpsxq"] * 4) / df["prccq"]

    # ------------------------
    # 6. Leverage / liquidity
    # ------------------------
    df["leverage"]      = df["ltq"] / df["atq"]
    df["current_ratio"] = df["actq"] / df["lctq"]
    df["cash_assets"]   = df["cheq"] / df["atq"]

    # ------------------------
    # 7. Accruals
    # ------------------------
    df["accruals_ta"] = (df["ni_ttm"] - df["oancf_ttm"]) / df["atq"]

    # ------------------------
    # Clean & keep only useful columns
    # ------------------------
    feature_cols = [
        "sales_growth_qoq", "sales_growth_ttm", "asset_growth", "equity_growth",
        "roa_ttm", "roe_ttm", "gross_margin_ttm", "oper_margin_ttm",
        "net_margin_ttm", "log_mktcap", "bm", "earnings_yield", "cf_yield",
        "sales_yield", "div_yield", "leverage", "current_ratio", "cash_assets",
        "accruals_ta"
    ]

    final_df = df[["tic", "datadate"] + feature_cols].copy()
    final_df = final_df.sort_values(["tic", "datadate"]).reset_index(drop=True)

    return final_df


In [3]:
features = build_fundamental_features(fundq)

In [4]:
# select 2014 - 2023

features = features[(features["datadate"].dt.year >= 2013) & (features["datadate"].dt.year <= 2023)].copy()

In [5]:
# save features
features.to_csv("../data/model/fundamental_features.csv", index=False)